# Basketball Scouting & Game-Plan Coordinator - progress demo
A LangGraph swarm of role-based agents (local `qwen3` 8B through Ollama) that reads
**offline January 2026 NBA data** and produces a 3-key game plan for the head coach.

In [1]:
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from scout import memory
from scout.data import load
from scout.graph import SPECIALISTS, build_graph, run
from scout.guardrails import check_matchup, validate_report
from scout.tools import ALL_TOOLS

## 1. Offline data (long-term knowledge) and load-time bounds checks

In [2]:
data = load()
print("teams:", len(data.team_codes))
print("games:", data.team_logs.GAME_ID.nunique())
print("player-games:", len(data.player_logs))
print("shots:", len(data.shots))
print("games with rotation data:", data.rotations.GAME_ID.nunique())
print("rows dropped by bounds checks:", data.dropped_rows)

teams: 30
games: 233
player-games: 4990
shots: 41454
games with rotation data: 233
rows dropped by bounds checks: {'team_logs': 0, 'player_logs': 0, 'shots': 0}


## 2. Tools - deterministic, return `found: False` instead of a guess

In [3]:
result = ALL_TOOLS["weakest_defenders"].invoke({"team": "DEN"})
print(json.dumps(result, indent=1)[:900])

{
 "found": true,
 "team": "DEN",
 "weakest": [
  {
   "player": "Peyton Watson",
   "position": "G",
   "games": 15,
   "contested_fga_per_game": 15.3,
   "fg_pct_allowed": 50.4,
   "shooters_normal_fg_pct": 46.9,
   "pct_plusminus": 3.5
  },
  {
   "player": "Tim Hardaway Jr.",
   "position": "G-F",
   "games": 15,
   "contested_fga_per_game": 8.3,
   "fg_pct_allowed": 49.6,
   "shooters_normal_fg_pct": 46.4,
   "pct_plusminus": 3.2
  },
  {
   "player": "Aaron Gordon",
   "position": "F",
   "games": 10,
   "contested_fga_per_game": 11.5,
   "fg_pct_allowed": 52.2,
   "shooters_normal_fg_pct": 49.1,
   "pct_plusminus": 3.0
  }
 ],
 "strongest": [
  {
   "player": "Christian Braun",
   "position": "G",
   "games": 3,
   "contested_fga_per_game": 10.3,
   "fg_pct_allowed": 35.5,
   "shooters_normal_fg_pct": 46.2,
   "pct_plusminus": -10.7
  },
  {
   "player": "Jalen Pickett",
   "posit


In [4]:
ALL_TOOLS["player_vs_opponent"].invoke({"player": "Shai Gilgeous-Alexander", "opponent": "DEN"})

{'found': False,
 'player': 'Shai Gilgeous-Alexander',
 'january_average': {'pts': 31.7, 'fg_pct': 54.1, 'fg3_pct': 30.4, 'tov': 2.4},
 'message': 'Shai Gilgeous-Alexander did not play DEN in January 2026. Do not invent head-to-head numbers; use team_shot_zones/team_defense_profile instead.'}

## 3. Guardrails
Input validation, and the grounding check that rejects numbers not present in tool output.

In [5]:
try:
    check_matchup("OKC", "XYZ")
except ValueError as error:
    print("Rejected:", str(error)[:90], "...")

Rejected: Unknown team code(s) ['XYZ']. Valid codes: ['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL ...


In [6]:
evidence = [json.dumps(ALL_TOOLS["team_summary"].invoke({"team": "DEN"}))]
made_up_report = (
    "FINDINGS:\n- DEN score 109.8 points per game but Jokic averages 31.4 points "
    "in 52 minutes, shooting 104% at the rim."
)
for problem in validate_report(made_up_report, evidence):
    print("-", problem)

- Numbers not found in any tool output: 104%, 31.4, 52
- '104%' is not a valid percentage


## 4. The graph (orchestration)
![Project architecture](../docs/architecture.svg)

In [7]:
print(build_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	input_guard(input_guard)
	validator(validator)
	head_coach(head_coach)
	shot_chart_analyst(shot_chart_analyst)
	star_coverage(star_coverage)
	defensive_coordinator(defensive_coordinator)
	offensive_architect(offensive_architect)
	__end__([<p>__end__</p>]):::last
	__start__ --> input_guard;
	defensive_coordinator --> validator;
	input_guard --> defensive_coordinator;
	input_guard --> offensive_architect;
	input_guard --> shot_chart_analyst;
	input_guard --> star_coverage;
	offensive_architect --> validator;
	shot_chart_analyst --> validator;
	star_coverage --> validator;
	validator -.-> defensive_coordinator;
	validator -.-> head_coach;
	validator -.-> offensive_architect;
	validator -.-> shot_chart_analyst;
	validator -.-> star_coverage;
	head_coach --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. End-to-end run: OKC scouting DEN

In [8]:
start = time.time()
state = run("OKC", "DEN")
print(f"finished in {time.time() - start:.0f}s")

finished in 84s


In [9]:
print("Tool calls made by the agents:")
for call in state["trace"]:
    print(f"  {call['role']:<22} {call['tool']:<20} {call['args']}  found={call['found']}")

print()
for role in SPECIALISTS:
    problems = state["problems"][role] or "none"
    print(f"{role:<22} attempts={state['attempts'][role]}  problems={problems}")

Tool calls made by the agents:
  defensive_coordinator  team_summary         {'team': 'DEN'}  found=True
  defensive_coordinator  top_scorers          {'n': 2, 'team': 'DEN'}  found=True
  defensive_coordinator  rotation_patterns    {'team': 'DEN'}  found=True
  offensive_architect    weakest_defenders    {'team': 'DEN'}  found=True
  offensive_architect    team_shot_zones      {'side': 'defense', 'team': 'DEN'}  found=True
  offensive_architect    team_defense_profile {'team': 'DEN'}  found=True
  offensive_architect    top_scorers          {'n': 3, 'team': 'OKC'}  found=True
  shot_chart_analyst     top_scorers          {'n': 2, 'team': 'DEN'}  found=True
  shot_chart_analyst     player_shot_profile  {'player': 'Jamal Murray'}  found=True
  shot_chart_analyst     player_shot_profile  {'player': 'Peyton Watson'}  found=True
  shot_chart_analyst     team_shot_zones      {'side': 'offense', 'team': 'DEN'}  found=True
  star_coverage          top_scorers          {'n': 2, 'team': 'OKC'} 

In [10]:
for role in SPECIALISTS:
    print(f"=== {role} ===")
    print(state["reports"][role], "\n")

=== shot_chart_analyst ===
FINDINGS:
- Jamal Murray is DEN's top scorer with 27.7 PPG, shooting 51.7% from the field and 43.0% from 3-point range.
- Peyton Watson is DEN's second top scorer with 21.9 PPG, shooting 49.0% from the field and 46.2% from 3-point range.
- Jamal Murray shoots 64.6% in The Paint (Non-RA), which is 20.2% better than the league average.
- Jamal Murray shoots 63.8% from Less Than 8 ft., which is 30.0% of his total attempts.
- Jamal Murray shoots 67.5% from Restricted Area, which is 0.4% better than the league average.
- Peyton Watson shoots 63.3% from Restricted Area, which is 3.8% worse than the league average.
- DEN shoots 65.0% from Restricted Area, which is 2.1% worse than the league average.
- DEN shoots 37.5% from Above the Break 3, which is 2.8% better than the league average.
- DEN shoots 54.0% from Left Corner 3, which is 14.9% better than the league average.
- DEN shoots 33.8% from Right Corner 3, which is 3.8% worse than the league average.

RECOMMENDA

In [11]:
print(state["final_plan"])
print("\nPlan validation problems:", state["plan_problems"] or "none")

GAME PLAN: OKC vs DEN  
KEY 1 (Defense): Focus on defending Jamal Murray's mid-range and restricted area shots, as he is efficient in these zones. - Why: Jamal Murray shoots 63.8% from Less Than 8 ft., which is 30.0% of his total attempts, and 67.5% from Restricted Area, which is 0.4% better than the league average.  
KEY 2 (Offense): Attack the Restricted Area by using Chet Holmgren's inside presence to create high-percentage shots. - Why: DEN allows opponents to shoot 71.2% in Restricted Area, which is 4.1% better than league average.  
KEY 3 (Stars & Rotation): Use a switch-heavy defense to disrupt Jamal Murray's off-ball movement and limit his open looks from above the break 3. - Why: Jamal Murray shoots 37.5% from Above the Break 3, which is 2.8% better than the league average.  

WATCH-OUTS: Watch Jamal Murray's efficiency in the paint (64.6% in The Paint) and his frequent closing role (56.2%). Also, note that DEN's left corner 3 shooting is strong (54.0%, 14.9% better than leagu

## 6. Memory - plans persist and are fed back to the Head Coach next time

In [12]:
print(json.dumps(memory.last_plan("OKC", "DEN"), indent=1)[:600])

{
 "time": "2026-11-08T20:03:26",
 "my_team": "OKC",
 "opponent": "DEN",
 "validated": true,
 "plan": "GAME PLAN: OKC vs DEN  \nKEY 1 (Defense): Focus on defending Jamal Murray's mid-range and restricted area shots, as he is efficient in these zones. - Why: Jamal Murray shoots 63.8% from Less Than 8 ft., which is 30.0% of his total attempts, and 67.5% from Restricted Area, which is 0.4% better than the league average.  \nKEY 2 (Offense): Attack the Restricted Area by using Chet Holmgren's inside presence to create high-percentage shots. - Why: DEN allows opponents to shoot 71.2% in Restricted 
